# 09 — The propositions, participant round (§4.3)

C02, for the real round. Run it **after the round closes**, and only after the
gate (C03) has passed. It is the same analysis as `06_analysis`, which was
validated on the pilot. The parameters below are fixed by D96–D102; change
them only with a `DECISIONS.md` entry. Every section says "no data yet" rather
than failing while the round is empty.

Sections, in §4.3's order:

1. **Descriptives.** Sessions, request types and shapes, and the two feedback
   axes (Table 12 as two axes, D102).
2. **P2: execution failures.** Rates for both rounds with 95 % Wilson
   intervals, failure types, and the rank agreement (Kendall's τ-b,
   descriptive, D101).
3. **P1: coverage.** Share of participant requests (D100) and turns whose D24
   signature the simulated runs produced, where uncovered ones concentrate
   (shapes7, D99), and why each missed.
4. **P3: judge vs feedback.** One-sided Mann–Whitney U with the rank-biserial r,
   thumbs and accept/reject separately (D101). Blocked on Q35.
5. **Usability.** UMUX-LITE, SUS equivalent, curved grade, recommend item.
6. **Recovered tool errors.** Secondary reading, Q36 option C.

The logic lives in `src/analysis.py` (tested in `tests/test_analysis.py`), the
figures in `src/figures.py`.

**Privacy.** This notebook reads participant rows. Outputs are stripped on
commit (`notebooks/study/.gitattributes`). Nothing prints a participant's
words, an id or an email. Figures go to `figures/<run_type>/`, which is
git-ignored; they reach the thesis by being copied there on purpose.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))

import matplotlib.pyplot as plt
import pandas as pd

from src import analysis, db, figures, pilot, shapes
from src.execution import _place_ids_of, latest_attempts
from src.signature import SIGNATURE_RULE_VERSION

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)

# --- parameters -------------------------------------------------------------
RUN_TYPE = "study"
SIM_RUN_IDS = [
    "1a872c09-7070-4789-a806-904969f9185d",  # B10 baseline run, adopted (D96)
    "e097f78e-3326-4aea-acaa-0b904d3b0a1c",  # B11 regression run
]
ALPHA = 0.05  # P3 only; P2's τ is descriptive (D101)
save = figures.Saver(RUN_TYPE, enabled=True)
figures.setup()

def rate_frame(rates):
    return pd.DataFrame([vars(r) for r in rates], columns=["group", "n", "failures", "rate", "ci_low", "ci_high"]).set_index("group")

def side_by_side(sim_rates, part_rates):
    """Both rounds' rates per class, one row per class, percentages formatted."""
    t = rate_frame(sim_rates).join(rate_frame(part_rates), lsuffix=" sim", rsuffix=" part", how="outer")
    pct = {c: "{:.1%}" for c in t.columns if c.split()[0] in ("rate", "ci_low", "ci_high")}
    return t.style.format(pct, na_rep="–").format({c: "{:.0f}" for c in t.columns if c not in pct}, na_rep="–")

def rate_figure(rounds, title, xlabel, name):
    n = len({r.group for rates in rounds.values() for r in rates})
    fig, ax = plt.subplots(figsize=(6.4, 0.5 * max(n, 1) + 1.2))
    figures.ci_plot(ax, rounds, title, xlabel)
    save(fig, name); plt.show()

client = db.admin()
data = pilot.load(client, RUN_TYPE)
turns = list(data.suggestions)
replies = [t["reply_message_id"] for t in turns if t.get("reply_message_id")]
refs = {m["id"]: _place_ids_of(m.get("metadata"))
        for m in (client.table("chat_messages").select("id, metadata").in_("id", replies).execute().data
                  if replies else [])}
requests = [seg for run in analysis.runs_of(turns) for seg in analysis.segments(run, refs)]

sim_executions, superseded = analysis.simulated_executions(client, SIM_RUN_IDS)
sim_turns = [t for ex in sim_executions for t in ex]

print(f"participant ({RUN_TYPE}): {len(data.sessions)} sessions, {len(turns)} turns, {len(requests)} requests")
print(f"simulation: {len(SIM_RUN_IDS)} runs, {len(sim_executions)} executions, {len(sim_turns)} turns "
      f"({superseded} superseded attempts left out, D53)")
print(f"rules: signature {SIGNATURE_RULE_VERSION}, shape {shapes.REQUEST_SHAPE_RULE_VERSION}, "
      f"analysis {analysis.ANALYSIS_VERSION}")

## 1. Descriptives

Sessions are pseudonymous (`S1, …` in consent order, `D1, …` declined).
Requests are Tonic runs split under D100.

In [ ]:
sessions = pd.DataFrame(pilot.session_table(data))
if len(sessions):
    consented = sessions[sessions.status == "consented"]
    print(f"consented sessions: {len(consented)}; turns per session: median {consented.turns.median()}, "
          f"range {consented.turns.min()}–{consented.turns.max()}")
    display(sessions)
else:
    print("no sessions yet")

In [ ]:
dist = pd.DataFrame({
    "participant turns": pd.Series([t.get("request_type") or "(none)" for t in turns], dtype=object).value_counts(),
    "simulated turns": pd.Series([t.get("request_type") or "(none)" for t in sim_turns], dtype=object).value_counts(),
}).fillna(0).astype(int)
shape_dist = pd.DataFrame({
    "participant requests": pd.Series([analysis.interaction_shape(r) for r in requests], dtype=object).value_counts(),
    "participant turns": pd.Series([analysis.turn_shape(t) for t in turns], dtype=object).value_counts(),
    "simulated turns": pd.Series([analysis.turn_shape(t) for t in sim_turns], dtype=object).value_counts(),
}).fillna(0).astype(int)
display(dist)
shape_dist

In [ ]:
# Figure 1 — shape distribution, share of turns per round (shapes7, D99).
def shares(col):
    total = shape_dist[col].sum()
    return (shape_dist[col] / total).to_dict() if total else {}

fig, ax = plt.subplots(figsize=(6.4, 0.42 * max(len(shape_dist), 1) + 1))
figures.paired_shares(ax, {"simulation": shares("simulated turns"), "participant": shares("participant turns")},
                      f"Request shapes (n = {len(sim_turns)} simulated, {len(turns)} participant turns)",
                      "share of agent turns")
save(fig, "fig1_shapes"); plt.show()

**Table 12 as two axes (D102).** The structural verdict exists only in
simulation (D73) and `user_action` only for participants, so they are reported
side by side, not crossed. §4.3 (1): if `ignored` dominates, it is said here
before anything is interpreted.

In [ ]:
axes = analysis.participant_axes(turns, data.feedback)
ua = pd.Series(axes["user_action"], dtype=int).sort_values(ascending=False)
exec_rows = []
for rid in SIM_RUN_IDS:
    rows = client.table("study_case_executions").select("*").eq("run_id", rid).execute().data
    exec_rows += latest_attempts(rows)[0]
structural = pd.Series([r.get("structural_outcome") or "(not checked)" for r in exec_rows], dtype=object).value_counts()

print("participant — user_action per turn (n = %d):" % ua.sum()); print(ua.to_string() if len(ua) else "  (none)")
if len(ua) and ua.get("ignored", 0) > ua.sum() / 2:
    print("\n>>> ignored dominates: most suggestions got no reaction; read the agreement measures with that in mind.")
print("\nparticipant — thumbs (n = %d):" % sum(axes["thumbs"].values()))
print(pd.Series(axes["thumbs"], dtype=int).to_string() if axes["thumbs"] else "  (none)")
print("\nsimulation — structural outcome per execution (n = %d):" % structural.sum()); print(structural.to_string())

## 2. P2: execution-failure rates

An execution failure is an agent turn with `error_code` set, read from the same
column in both rounds. Simulated turns come from the latest attempt of each
execution (D53). Rates carry 95 % Wilson intervals (`ci_low`, `ci_high`). The
primary claim is the **rank** correspondence across request types; Kendall's
τ-b is reported as an effect size with no p-value (D101).

In [ ]:
overall = pd.DataFrame([vars(analysis.overall_rate(sim_turns)), vars(analysis.overall_rate(turns))],
                       index=["simulation", "participant"]).drop(columns="group")
sim_rates, part_rates = analysis.failure_rates(sim_turns), analysis.failure_rates(turns)
display(overall.style.format({"rate": "{:.1%}", "ci_low": "{:.1%}", "ci_high": "{:.1%}"}, na_rep="–"))
side_by_side(sim_rates, part_rates)

In [ ]:
agreement = analysis.rank_agreement(sim_rates, part_rates)
print(f"Kendall τ-b over {agreement['n_classes']} request types {agreement['classes']}: "
      + (f"{agreement['tau_b']:+.2f}" if agreement["tau_b"] is not None else agreement["note"]))
print("(descriptive, D101 — no p-value: at this number of classes no rank test can reach significance)")

types = pd.DataFrame({
    "simulation": pd.Series(analysis.failure_types(sim_turns), dtype=int),
    "participant": pd.Series(analysis.failure_types(turns), dtype=int),
}).fillna(0).astype(int)
types if len(types) else "no failed turn in either round"

In [ ]:
# Figure 2 — failure rate per request type, both rounds, 95 % Wilson intervals.
tau = agreement["tau_b"]
rate_figure({"simulation": sim_rates, "participant": part_rates},
            "Execution failures by request type" + (f" — Kendall τ-b = {tau:+.2f}" if tau is not None else ""),
            "execution-failure rate (95 % Wilson interval)", "fig2_failure_rates")

## 3. P1: coverage

A participant request is **covered** when a simulated execution of the runs
above has the same D24 signature: tool sequence with consecutive repeats
collapsed, request type, bucketed calls per tool and turns, and four parameter
flags. Coverage is also reported per turn, since no claim may rest on a
request count (D100). §4.2.3: the uncovered interactions are of more interest
than the proportion.

In [ ]:
cov_requests = analysis.coverage(requests, sim_executions, "request")
cov_turns = analysis.coverage([[t] for t in turns], sim_executions, "turn")
summary = pd.DataFrame({"per request": analysis.coverage_summary(cov_requests),
                        "per turn": analysis.coverage_summary(cov_turns)}).T
display(summary.style.format({"share": "{:.1%}", "ci_low": "{:.1%}", "ci_high": "{:.1%}"}, na_rep="–"))

unc = pd.DataFrame({
    "uncovered requests": pd.Series(analysis.uncovered_by(cov_requests), dtype=int),
    "all requests": pd.Series([r.shape for r in cov_requests], dtype=object).value_counts(),
}).fillna(0).astype(int).sort_values("uncovered requests", ascending=False)
unc if len(unc) else "no participant requests yet"

**Why each uncovered request missed.** Every uncovered request against the
closest signature the simulated runs produced, and the parts in which they
differ. One part off, especially `turns` or `calls per tool`, is a near miss:
the suite does this, but not in exactly this way, and D24's matching
tolerance counts it as uncovered. Several parts off, or a `tool sequence`
difference, is a shape the suite does not produce. This is a diagnostic only:
it changes nothing about what is covered. Only tool names and labels are shown,
never request text.

In [ ]:
diagnosis = pd.DataFrame(analysis.uncovered_diagnosis(requests, sim_executions))
if len(diagnosis):
    near = (diagnosis["parts off"] == 1).sum()
    print(f"{len(diagnosis)} uncovered requests: {near} one part off (near miss), "
          f"{len(diagnosis) - near} further off")
    display(diagnosis)
    display(pd.Series([p for d in diagnosis["differs in"] for p in d.split(", ")])
            .value_counts().rename("uncovered requests differing in this part").to_frame())
else:
    print("no uncovered participant request" + ("" if requests else " (no requests yet)"))

In [ ]:
# Figure 3 — covered and uncovered requests per shape.
frame = pd.DataFrame([vars(r) for r in cov_requests])
fig, ax = plt.subplots(figsize=(6.4, 0.42 * max(frame["shape"].nunique() if len(frame) else 1, 1) + 1))
if len(frame):
    counts = frame.groupby(["shape", "covered"]).size().unstack(fill_value=0).reindex(columns=[True, False], fill_value=0)
    counts = counts.loc[counts.sum(axis=1).sort_values().index]
    ax.barh(counts.index, counts[True], color=figures.SERIES, label="covered by the suite", height=0.6)
    ax.barh(counts.index, counts[False], left=counts[True] + 0.04, color="#c3c2b7", label="uncovered",
            height=0.6, hatch="//", edgecolor="white", linewidth=0)
    for i, (c, u) in enumerate(zip(counts[True], counts[False])):
        ax.annotate(f"{c}/{c + u}", (c + u, i), xytext=(6, 0), textcoords="offset points", va="center",
                    fontsize=8, color=figures.MUTED)
    ax.set_xlabel("participant requests")
    ax.grid(axis="x", color=figures.GRID, linewidth=0.8); ax.set_axisbelow(True)
    ax.legend(frameon=False, loc="lower right")
    s = analysis.coverage_summary(cov_requests)
    ax.set_title(f"Coverage: {s['covered']}/{s['n']} requests ({s['share']:.0%}) match a simulated signature")
else:
    ax.set_title("Coverage"); figures.empty(ax)
save(fig, "fig3_coverage"); plt.show()

## 4. P3: judge score vs user feedback

The judge scores each participant reply blind to what the member did, and
the scores are split by thumbs and by accept/reject, never pooled.
`ignored` and unset actions belong to neither group.

**Blocked on Q35.** No judge scores exist for participant replies yet, and the
judge cannot run on them as built: its rubrics are per test case, not per
request type; it prepends the fixture trip, not the member's trip; and
`study_scores` is keyed to a simulated execution. `PARTICIPANT_SCORES` below
stays empty until Q35 is ruled. The test cells run as soon as it is filled,
and report *too few* until each group holds two scores.

In [ ]:
PARTICIPANT_SCORES: dict[str, float] = {}  # turn id -> judge score; see Q35

groups = analysis.feedback_groups(turns, data.feedback, PARTICIPANT_SCORES)
p3 = pd.DataFrame({signal: analysis.mann_whitney(g["positive"], g["negative"]) for signal, g in groups.items()}).T
if "p" in p3:
    p3["reject H0 at α"] = p3["p"] < ALPHA
print(f"{len(PARTICIPANT_SCORES)} participant replies scored")
p3

In [ ]:
# Figure 4 — judge score by feedback group, one panel per signal.
if PARTICIPANT_SCORES:
    fig, axs = plt.subplots(1, 2, figsize=(6.4, 3.2), sharey=True)
    for ax, (signal, g) in zip(axs, groups.items()):
        for x, key in enumerate(("positive", "negative")):
            vals = g[key]
            ax.scatter([x + (k % 5 - 2) * 0.04 for k in range(len(vals))], vals, s=36,
                       color=figures.ROUND_COLOR["participant"], edgecolor="white", linewidth=1.5, zorder=3)
            if vals:
                ax.plot([x - 0.25, x + 0.25], [pd.Series(vals).median()] * 2, color=figures.INK, linewidth=2)
        ax.set_xticks([0, 1], [f"positive\n(n = {len(g['positive'])})", f"negative\n(n = {len(g['negative'])})"])
        ax.set_title(signal); ax.set_ylim(0.5, 10.5)
    axs[0].set_ylabel("judge score (1–10)")
    save(fig, "fig4_judge_vs_feedback"); plt.show()
else:
    print("no participant judge scores yet — figure 4 is drawn once there are (Q35)")

## 5. Usability

UMUX-LITE on 0–100 (D18), then the **UMUX-LITE → SUS conversion** (Lewis,
Utesch & Maher, 2013): SUS-equivalent ≈ 0.65 × UMUX-LITE + 22.9, an estimate
of the SUS score rather than a measured one. It is graded on the Sauro–Lewis
curved grading scale. The recommend item is on 1–7.

In [ ]:
usab = pd.DataFrame(analysis.usability(data.post_task))
if len(usab):
    mean_sus = usab.sus_equivalent.mean()
    print(f"n = {len(usab)} submitted surveys; mean UMUX-LITE {usab.umux_lite.mean():.1f}, "
          f"SUS-equivalent {mean_sus:.1f} → grade {analysis.curved_grade(mean_sus)}; "
          f"recommend median {usab.recommend.median()}")
    display(usab.describe().loc[["count", "mean", "std", "min", "50%", "max"]].round(1))
    display(usab.grade.value_counts().rename("surveys").to_frame())
else:
    print("no submitted post-task survey yet")

In [ ]:
# Figure 5 — SUS-equivalent scores against the curved grading scale.
fig, ax = plt.subplots(figsize=(6.4, 1.8))
figures.grade_band(ax, list(usab.sus_equivalent) if len(usab) else [], analysis.CURVED_GRADES,
                   "Usability", "SUS-equivalent score (from UMUX-LITE)", color=figures.ROUND_COLOR["participant"])
save(fig, "fig5_usability"); plt.show()

## 6. Recovered tool errors

A **recovered error** is a tool call that came back an error inside a turn that
still completed (no `error_code`). The agent got there, but not first time.
It is not an execution failure by §4.2.3's definition and not a pass
criterion (D30).

**Status: a secondary, descriptive reading (Q36 option C, not yet ruled).** It
is compared the way P2 is: a rate per class for both rounds with 95 % Wilson
intervals, and Kendall's τ-b over the request types both rounds observed, with
no p-value (D101's reasoning applies unchanged). Two grains:

- **per tool:** errored calls / calls of that tool in completed turns;
- **per request type:** completed turns with at least one errored call /
  completed turns of that type.

In [ ]:
sim_rec_tool, part_rec_tool = analysis.recovered_error_rates(sim_turns), analysis.recovered_error_rates(turns)
sim_rec_type = analysis.recovered_error_rates(sim_turns, key="request_type")
part_rec_type = analysis.recovered_error_rates(turns, key="request_type")
rec_agreement = analysis.rank_agreement(sim_rec_type, part_rec_type)
print(f"Kendall τ-b over {rec_agreement['n_classes']} request types {rec_agreement['classes']}: "
      + (f"{rec_agreement['tau_b']:+.2f}" if rec_agreement["tau_b"] is not None else rec_agreement["note"]))
print("\nper tool:")
display(side_by_side(sim_rec_tool, part_rec_tool))
print("per request type:")
side_by_side(sim_rec_type, part_rec_type)

In [ ]:
# Figure 6 — recovered tool errors per tool, both rounds, 95 % Wilson intervals.
# Only tools that errored at least once in either round; the rest are in the table above.
errored = {r.group for r in sim_rec_tool + part_rec_tool if r.failures}
rate_figure({"simulation": [r for r in sim_rec_tool if r.group in errored],
             "participant": [r for r in part_rec_tool if r.group in errored]},
            "Recovered tool errors by tool", "share of calls that came back an error (95 % Wilson interval)",
            "fig6_recovered_errors")

In [ ]:
print(f"figures written to {save.dir.resolve()}")